# Concrete Evidence: train the rebar segmentation model (YOLOv8-seg)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

- **Part A, baseline:** pre-train on **ConRebSeg** (real construction sites, CC BY 4.0, DTU). Only its self-collected photos; the YouTube frames are skipped (no clear licence). Class: `bar`.
- **Part B, fine-tune:** on our own Roboflow labels (`bar · tie · hook_90 · hook_135`), same split and seed, then compare with the baseline.

Column Tie Detection and SteelBindings (Roboflow, CC BY 4.0) have **boxes only**, no outlines, so they can't train this segmentation model. They'd need a separate tie detector.

Credit: Schmidt & Nalpantidis (2025), *Segmentation dataset for reinforced concrete construction*, Automation in Construction 171. Data DOI 10.11583/DTU.26213762.

In [ ]:
# 1. GPU + install (same versions as requirements.txt)
!nvidia-smi -L
!pip install -q ultralytics==8.4.216 roboflow==1.7.3

In [ ]:
# 2. Results go to Google Drive, so best.pt survives the session
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/ConcreteEvidence/runs'
import os; os.makedirs(SAVE, exist_ok=True)
SEED = 0  # same seed for baseline and fine-tune, so the two compare

In [ ]:
# 3. The repo code
%cd /content
!rm -rf AAD_Research_Studio_03 && git clone -q https://github.com/iaac-maai/AAD_Research_Studio_03
%cd AAD_Research_Studio_03

## Part A: baseline on ConRebSeg

The download is 22.6 GB straight to Colab's disk (not your Drive), a few minutes. Every file is checked against the checksums published with the dataset.

In [ ]:
# 4. Download ConRebSeg from figshare (DTU Data runs on figshare) and check it
import hashlib, subprocess
D = 'data/conrebseg'; os.makedirs(D, exist_ok=True)
FILES = {47627011: 'metadata.json', 47627014: 'samples.json', 47836036: 'ConRebSeg.zip'}
MD5 = {'metadata.json': '8024f05766f52549abfbf6e869679ce3', 'samples.json': '3a7ca34b89077fd6104a4fa8a5aba0b5',
       'ConRebSeg.zip': 'c1517d9e41fc59cc7be2835b3e2f4642'}
for fid, name in FILES.items():
    subprocess.run(['curl', '-sSL', '--retry', '10', '-C', '-', '-o', f'{D}/{name}',
                    f'https://ndownloader.figshare.com/files/{fid}'], check=True)
    h = hashlib.md5()
    with open(f'{D}/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''): h.update(block)
    assert h.hexdigest() == MD5[name], f'{name} is damaged: download it again'
    print('ok', name)

In [ ]:
# 5. Unpack the site photos (no YouTube) and convert to YOLO-seg. LIMIT = None uses all 12,000.
LIMIT = None  # e.g. 1000 for a quick first run
!python pipeline.py conrebseg {'--limit ' + str(LIMIT) if LIMIT else ''}
!rm -f data/conrebseg/ConRebSeg.zip
!cat data/conrebseg_yolo/data.yaml

In [ ]:
# 6. Look at one labelled image before training
import glob, random, cv2, numpy as np, matplotlib.pyplot as plt
img = random.choice(glob.glob('data/conrebseg_yolo/images/train/*'))
lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
im = cv2.cvtColor(cv2.imread(img), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
for line in open(lbl).read().splitlines():
    if not line: continue
    v = list(map(float, line.split())); pts = (np.array(v[1:]).reshape(-1, 2) * [w, h]).astype(np.int32)
    cv2.polylines(im, [pts], True, (255, 0, 0), 3)
plt.figure(figsize=(12, 8)); plt.imshow(im); plt.axis('off'); plt.title(os.path.basename(img))

In [ ]:
# 7. TRAIN the baseline. yolov8n-seg, 640 px, 20 epochs: roughly 40-60 min on a free T4.
from ultralytics import YOLO
EPOCHS, IMGSZ = 20, 640
model = YOLO('yolov8n-seg.pt')
model.train(data='data/conrebseg_yolo/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=8,
            seed=SEED, deterministic=True, max_det=500, project=SAVE, name='conrebseg_baseline', exist_ok=True)

In [ ]:
# 8. Score on the test split (never seen in training) -> numbers for the board and the slides
run = f'{SAVE}/conrebseg_baseline'
best = YOLO(f'{run}/weights/best.pt')
m = best.val(data='data/conrebseg_yolo/data.yaml', split='test', imgsz=IMGSZ, max_det=500, plots=False)
print(f'mask mAP50 {m.seg.map50:.4f}   precision {m.seg.mp:.4f}   recall {m.seg.mr:.4f}')
print('Log this run on the team board: run conrebseg_baseline, data ConRebSeg', LIMIT or 'all', 'images')
for r in best.predict(sorted(glob.glob('data/conrebseg_yolo/images/test/*'))[:3], imgsz=IMGSZ, conf=0.25, max_det=500):
    plt.figure(figsize=(12, 8)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')

## Part B: fine-tune on our own labels

In Roboflow: **Generate** a version (70/20/10 split, blur + brightness augmentation). The key is typed into the box below and never saved in the notebook.

In [ ]:
# 9. Fine-tune from the ConRebSeg baseline on our labelled frames, same seed
from getpass import getpass
from roboflow import Roboflow
VERSION = 1
rf = Roboflow(api_key=getpass('Roboflow API key: '))
proj = rf.workspace('akshat-dayal-students-iaac-net').project('concrete-evidence-column-cages')
ds = proj.version(VERSION).download('yolov8', location='data/ours', overwrite=True)
ft = YOLO(f'{SAVE}/conrebseg_baseline/weights/best.pt')
ft.train(data='data/ours/data.yaml', epochs=100, imgsz=IMGSZ, batch=8, patience=20, seed=SEED,
         deterministic=True, max_det=500, project=SAVE, name='ours_finetune', exist_ok=True)
m = YOLO(f'{SAVE}/ours_finetune/weights/best.pt').val(data='data/ours/data.yaml', split='test', imgsz=IMGSZ, plots=False)
print(f'fine-tuned  mask mAP50 {m.seg.map50:.4f}   precision {m.seg.mp:.4f}   recall {m.seg.mr:.4f}')